# grid_100x100 PIE パトロール（A* + SpotDog 往復）

**前提**: UE Editor で `grid_100x100` を開き **PIE 実行中** にこのノートを実行してください。

## シナリオ

1. 外周ブロックを実体化 (`SetBlocking True`)
2. Humanoid をマス **(5, 5)**、SpotDog を **(10, 5)** にスポーン
3. SpotDog が A* 経路で **(50, 50)** へ移動 → **5 秒停止**
4. 出発点 **(10, 5)** へ A* で帰還

ロジック: `grid_env_10k_pie_patrol.py`（コストマップ + `path_planning_costmap` の格子 A*）

カーネル: `conda activate simworld`

## 初回セットアップ（Editor PIE）

1. **Robot_Dog** を Editor プロジェクトへコピー（未導入時）:
   `bash dev/grid_env_10k/scripts/install_robot_dog_editor.sh`
2. UE Editor を再起動するか、**PIE を Stop → Play** してアセットを再読込
3. Outliner の `block_*` はラベル名。UnrealCV は `BP_TransparentCube_C_UAID_...` を使うため、本スクリプトが位置から自動解決します

In [ ]:
import importlib
import sys
from pathlib import Path


def _find_project_root() -> Path:
    for start in (Path.cwd().resolve(), Path(".").resolve()):
        for candidate in (start, *start.parents):
            if (candidate / "setup.py").exists() and (candidate / "simworld").is_dir():
                return candidate
    return Path.cwd().resolve().parent.parent


_root = _find_project_root()
_g10k = _root / "dev" / "grid_env_10k"
for p in (_root, _g10k):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import grid_env_10k_pie_patrol as patrol

importlib.reload(patrol)
print(f"[Paths] root={_root}")

In [ ]:
# マス番号 (gx, gy) — 1 始まり
HUMAN_CELL = (5, 5)
ROBOT_START = (10, 5)
ROBOT_GOAL = (50, 50)
GOAL_DWELL_S = 5.0

print(
    f"human={HUMAN_CELL}, robot {ROBOT_START} -> {ROBOT_GOAL}, dwell={GOAL_DWELL_S}s"
)

In [ ]:
result = patrol.run_patrol_scenario(
    human_cell=HUMAN_CELL,
    robot_start_cell=ROBOT_START,
    robot_goal_cell=ROBOT_GOAL,
    goal_dwell_s=GOAL_DWELL_S,
)
result

In [ ]:
success = (
    result.perimeter_ok
    and result.robot_spawned
    and result.outbound_arrived
    and result.return_arrived
    and result.return_dist_cm <= patrol.RETURN_ARRIVE_TOLERANCE_CM
)
print(f"SUCCESS={success}, return_dist={result.return_dist_cm:.1f} cm")